In [0]:
from pyspark.sql import functions as F

silver_match_details = spark.read.table("opendota.silver.match_details")
silver_match_player_details = spark.read.table("opendota.silver.match_player_details")

MAX_TOWERS_SIDE = 11
MAX_BARRACKS_SIDE = 6
MIN_GAMES = 10
MIN_PICKS = 20


def build_player_match_base():
    matches = (
        silver_match_details.select(
            "match_id",
            "league_id",
            "region",
            "patch",
            "duration",
            "radiant_win",
            "radiant_team_id",
            "dire_team_id",
            "radiant_name",
            "dire_name",
            F.col("tower_status_radiant").alias("towers_standing_radiant"),
            F.col("tower_status_dire").alias("towers_standing_dire"),
            F.col("barracks_status_radiant").alias("barracks_standing_radiant"),
            F.col("barracks_status_dire").alias("barracks_standing_dire")
        )
    )

    return (
        silver_match_player_details.alias("p")
        .join(matches.alias("m"), on="match_id", how="inner")
        .filter(F.col("m.radiant_win").isNotNull())
        .withColumn("won", F.col("p.win").cast("int"))
        .withColumn(
            "team_id",
            F.when(
                F.col("p.is_radiant"),
                F.col("m.radiant_team_id"),
            ).otherwise(F.col("m.dire_team_id")),
        )
        .withColumn(
            "team_name",
            F.when(
                F.col("p.is_radiant"),
                F.col("m.radiant_name"),
            ).otherwise(F.col("m.dire_name")),
        )
        .withColumn(
            "towers_destroyed_by_team",
            F.when(
                F.col("p.is_radiant"),
                F.lit(MAX_TOWERS_SIDE) - F.col("m.towers_standing_dire"),
            ).otherwise(
                F.lit(MAX_TOWERS_SIDE) - F.col("m.towers_standing_radiant")
            ),
        )
        .withColumn(
            "barracks_destroyed_by_team",
            F.when(
                F.col("p.is_radiant"),
                F.lit(MAX_BARRACKS_SIDE) - F.col("m.barracks_standing_dire"),
            ).otherwise(
                F.lit(MAX_BARRACKS_SIDE) - F.col("m.barracks_standing_radiant")
            ),
        )
    )


def build_team_match_flat(include_side = False):
    valid_data = silver_match_details.filter(F.col("radiant_win").isNotNull())

    radiant_columns = [
        "match_id", "region", "patch", "duration", "league_id",
        F.col("radiant_team_id").alias("team_id"),
        F.col("radiant_name").alias("team_name"),
        F.col("radiant_win").cast("int").alias("won"),
        (F.lit(MAX_TOWERS_SIDE) - F.col("tower_status_dire")).alias("towers_destroyed"),
        (F.lit(MAX_BARRACKS_SIDE) - F.col("barracks_status_dire")).alias("barracks_destroyed")
    ]

    dire_columns = [
        "match_id", "region", "patch", "duration", "league_id",
        F.col("dire_team_id").alias("team_id"),
        F.col("dire_name").alias("team_name"),
        (1 - F.col("radiant_win").cast("int")).alias("won"),
        (F.lit(MAX_TOWERS_SIDE) - F.col("tower_status_radiant")).alias("towers_destroyed"),
        (F.lit(MAX_BARRACKS_SIDE) - F.col("barracks_status_radiant")).alias("barracks_destroyed"),
    ]

    if include_side:
        radiant_columns.append(F.lit(True).alias("is_radiant"))
        dire_columns.append(F.lit(False).alias("is_radiant"))
    
    radiant = valid_data.select(*radiant_columns)
    dire = valid_data.select(*dire_columns)

    return radiant.unionByName(dire)


def wilson_score_lower(wins, games, z = 1.96):
    prop = wins / games
    denom = 1 + z**2 / games
    center = prop + z**2 / (2 * games)
    margin = z * F.sqrt(prop * (1 - prop) / games + z**2 / (4 * games**2))

    return (center - margin) / denom

In [0]:
def first_blood_impact():
    return (
        build_player_match_base()
        .groupby("firstblood_claimed")
        .agg(
            F.count("*").alias("qty_players"),
            F.round(F.avg("won"), 4).alias("win_rate")
        )
        .orderBy("firstblood_claimed")
    )


def lane_efficiency_by_winrate():
    df = build_player_match_base().withColumn(
        "lane_efficiency_bucket",
        F.when(F.col("lane_efficiency") < 40, "< 40%")
        .when(F.col("lane_efficiency") < 55, "40-54%")
        .when(F.col("lane_efficiency") < 70, "55-69%")
        .otherwise("70%+")
    )

    return (
        df.groupBy("lane_efficiency_bucket")
        .agg(
            F.count("*").alias("player_count"),
            F.round(F.avg("won"), 4).alias("win_rate"),
            F.round(F.avg("lane_efficiency"), 2).alias("avg_lane_efficiency")
        )
        .orderBy("lane_efficiency_bucket")
    )


def kda_by_winrate():
    df = build_player_match_base().withColumn(
        "kda_bucket",
        F.when(F.col("kda") < 1, "< 1")
         .when(F.col("kda") < 2, "1-2")
         .when(F.col("kda") < 3, "2-3")
         .when(F.col("kda") < 5, "3-5")
         .otherwise("5+"),
    )

    return (
        df.groupBy("kda_bucket")
        .agg(
            F.count("*").alias("player_count"),
            F.round(F.avg("won"), 4).alias("win_rate"),
            F.round(F.avg("kda"), 2).alias("avg_kda"),
        )
        .orderBy("kda_bucket")
    )


def team_wilson_ranking():
    flat = (
        build_team_match_flat()
        .filter(F.col("team_id").isNotNull())
    )

    return (
        flat.groupBy("team_id", "team_name")
        .agg(
            F.count("*").alias("games"),
            F.sum("won").alias("wins"),
            F.round(F.avg("won"), 4).alias("win_rate")
        )
        .filter(F.col("games") >= MIN_GAMES)
        .withColumn(
            "wilson_score",
            F.round(wilson_score_lower(F.col("wins"), F.col("games")), 4)
        )
        .orderBy(F.desc("wilson_score"))
    )


def team_raw_wins_ranking(team_stats_df):
    return (
        team_stats_df
        .select("team_id", "team_name", "wins", "games")
        .orderBy(F.desc("wins"))
    )


def region_overview():
    flat = build_team_match_flat()

    by_team = (
        flat.groupBy("region", "team_id")
        .agg(F.avg("won").alias("team_wr"))
    )

    balance = (
        by_team.groupBy("region")
        .agg(F.round(F.stddev("team_wr"), 4).alias("team_winrate_stddev"))
    )

    players = (
        build_player_match_base()
        .groupBy("region")
        .agg(
            F.countDistinct("match_id").alias("match_count"),
            F.round(F.avg("kills_per_min"), 3).alias("avg_kills_per_min"),
            F.round(F.avg("gold_per_min"), 1).alias("avg_gpm"),
            F.round(F.avg("duration") / 60, 1).alias("avg_duration_min"),
        )
    )

    return (
        players.join(balance, on="region", how="left")
        .orderBy("region")
    )


def tournaments_by_region():
    return (
        silver_matches
        .groupBy("region", "league_id")
        .agg(
            F.count("*").alias("match_count"),
            F.round(F.avg("duration") / 60, 1).alias("avg_duration_min"),
        )
        .orderBy("region", F.desc("match_count"))
    )


def hero_meta_by_patch():
    return (
        build_player_match_base()
        .groupBy("patch", "hero_id")
        .agg(
            F.count("*").alias("picks"),
            F.round(F.avg("won"), 4).alias("win_rate"),
            F.round(F.avg("kda"), 2).alias("avg_kda"),
            F.round(F.avg("gold_per_min"), 1).alias("avg_gpm"),
        )
        .filter(F.col("picks") >= MIN_PICKS)
        .orderBy("patch", F.desc("win_rate"))
    )


def radiant_advantage_by_patch():
    return (
        silver_match_details
        .filter(F.col("radiant_win").isNotNull())
        .groupBy("patch")
        .agg(
            F.count("*").alias("match_count"),
            F.round(F.avg(F.col("radiant_win").cast("int")), 4).alias("radiant_win_rate"),
        )
        .orderBy("patch")
    )